In [1]:
# imports
import numpy as np
import pandas as pd
import random
from collections import deque, namedtuple
import torch
import torch.nn as nn
import torch.optim as optim
import joblib
import os

In [3]:
# configs

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

EMBED_FILE = "D:/ErgoSense/dataset/bilstm_embeddings.npz"   # embeddings produced earlier
SVM_MODEL = "D:/ErgoSense/models/svm_pipeline.pkl"          # to get classifier confidence if available
MODEL_OUT = "D:/ErgoSense/models/dqn_agent.pth"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


In [5]:
# Load embeddings saved for SVM (these are per-sequence embeddings)
data = np.load(EMBED_FILE)
X_train_emb = data['X_train']   # shape (N, emb_dim)
y_train = data['y_train']
X_val_emb = data['X_val']
y_val = data['y_val']

print("Train emb:", X_train_emb.shape, "Val emb:", X_val_emb.shape)

# We will construct simulated time-series episodes by chaining embeddings.
# For simplicity, treat each embedding as one time-step (you can also average sliding-window embeddings to make sessions).
# Build a list of episodes (each episode is sequence of (embedding, label)).
def build_episodes(embs, labels, episode_len=100):
    episodes = []
    n = len(embs)
    idx = 0
    while idx < n:
        end = min(idx + episode_len, n)
        episodes.append((embs[idx:end], labels[idx:end]))
        idx = end
    return episodes

train_episodes = build_episodes(X_train_emb, y_train, episode_len=200)
val_episodes = build_episodes(X_val_emb, y_val, episode_len=200)
print("Train episodes:", len(train_episodes), "Val episodes:", len(val_episodes))


Train emb: (3176, 256) Val emb: (794, 256)
Train episodes: 16 Val episodes: 4


In [ ]:
# Simple environment that streams an episode of embeddings and labels.
# State = [embedding (emb_dim), current_label (0/1), recent_bad_fraction (scalar), classifier_confidence (scalar)]
class SimEnv:
    def __init__(self, emb_seq, label_seq, window_bad=10, goal_hold=5):
        self.emb_seq = emb_seq
        self.label_seq = label_seq
        self.t = 0
        self.N = len(label_seq)
        self.window_bad = window_bad
        self.goal_hold = goal_hold  # how many frames need to be good to count success
        self.action_cooldown = 0
        self.last_action = 0

    def reset(self):
        self.t = 0
        self.action_cooldown = 0
        self.last_action = 0
        return self._get_state()

    def _get_recent_bad_fraction(self):
        start = max(0, self.t - self.window_bad)
        window = self.label_seq[start:self.t] if self.t>0 else np.array([])
        if len(window)==0:
            return 0.0
        return np.mean(window)

    def _get_state(self):
        emb = self.emb_seq[self.t]
        label = self.label_seq[self.t]
        recent_bad = self._get_recent_bad_fraction()
        # classifier_confidence: approximate via distance from boundary, use 0.8 for good, 0.6 for bad as placeholder
        cls_conf = 0.9 if label==0 else 0.7
        # state vector: concatenation
        state = np.concatenate([emb.flatten(), np.array([label, recent_bad, cls_conf])])
        return state.astype(np.float32)

    def step(self, action):
        """
        action: 0-noop,1-soft,2-strong,3-specific
        reward definition (simple):
         - per-step penalty if current label==1 (encourage fixing): -0.02
         - action penalty: -0.05 for any non-zero action (avoid spam)
         - if within next H steps label becomes 0 and stays 0 for goal_hold frames => +1 reward
         - if action ineffective for long => small negative
        """
        done = False
        cur_label = self.label_seq[self.t]
        reward = 0.0

        # per-step penalty for bad posture
        if cur_label == 1:
            reward -= 0.02

        # action cost
        if action != 0:
            reward -= 0.05


        # If the next few frames contain a transition to good posture, give positive reward.
        lookahead = 10
        future = self.label_seq[self.t+1 : min(self.N, self.t+1+lookahead)]
        success = False
        if len(future)>0 and np.all(future[:self.goal_hold] == 0):
            success = True

        if action != 0 and success:
            reward += 1.0  # successful corrective action

        # advance time
        self.t += 1
        if self.t >= self.N:
            done = True

        next_state = self._get_state() if not done else None
        info = {"success": success}
        return next_state, reward, done, info


In [9]:
# DQN network & replay buffer
class DQNNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )
    def forward(self, x):
        return self.net(x)

Transition = namedtuple('Transition', ('s', 'a', 'r', 's2', 'done'))

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buf = deque(maxlen=capacity)
    def push(self, *args):
        self.buf.append(Transition(*args))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        return Transition(*zip(*batch))
    def __len__(self):
        return len(self.buf)

# Agent wrapper
class DQNAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99, tau=0.005):
        self.q = DQNNet(state_dim, action_dim).to(DEVICE)
        self.target_q = DQNNet(state_dim, action_dim).to(DEVICE)
        self.target_q.load_state_dict(self.q.state_dict())
        self.opt = optim.Adam(self.q.parameters(), lr=lr)
        self.gamma = gamma
        self.tau = tau
        self.action_dim = action_dim
        self.eps = 1.0  # epsilon for epsilon-greedy

    def select_action(self, state):
        if random.random() < self.eps:
            return random.randrange(self.action_dim)
        s = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            qvals = self.q(s)
        return int(qvals.argmax().item())

    def update(self, batch):
        s = torch.tensor(np.stack(batch.s), dtype=torch.float32, device=DEVICE)
        a = torch.tensor(batch.a, dtype=torch.long, device=DEVICE).unsqueeze(1)
        r = torch.tensor(batch.r, dtype=torch.float32, device=DEVICE).unsqueeze(1)
        s2 = torch.tensor(np.stack([np.zeros_like(batch.s[0]) if x is None else x for x in batch.s2]),
                          dtype=torch.float32, device=DEVICE)
        done = torch.tensor(batch.done, dtype=torch.float32, device=DEVICE).unsqueeze(1)

        q = self.q(s).gather(1, a)
        with torch.no_grad():
            qnext = self.target_q(s2).max(1)[0].unsqueeze(1)
            qtarget = r + (1 - done) * self.gamma * qnext

        loss = nn.functional.mse_loss(q, qtarget)
        self.opt.zero_grad(); loss.backward(); self.opt.step()

        # soft update of target
        for tparam, param in zip(self.target_q.parameters(), self.q.parameters()):
            tparam.data.copy_(tparam.data * (1.0 - self.tau) + param.data * self.tau)
        return loss.item()


In [11]:
# Training loop
state_dim = X_train_emb.shape[1] + 3   # emb_dim + (label, recent_bad, conf)
action_dim = 4
agent = DQNAgent(state_dim, action_dim, lr=1e-3)
buffer = ReplayBuffer(20000)

NUM_EPOCHS = 30
BATCH = 64
MAX_STEPS_PER_EP = 200

for epoch in range(NUM_EPOCHS):
    total_reward = 0
    losses = []
    # iterate episodes
    for ep_idx, (emb_seq, lab_seq) in enumerate(train_episodes):
        env = SimEnv(emb_seq, lab_seq, window_bad=10, goal_hold=3)
        s = env.reset()
        ep_reward = 0
        steps = 0
        while True:
            a = agent.select_action(s)
            s2, r, done, info = env.step(a)
            buffer.push(s, a, r, s2, done)
            ep_reward += r
            s = s2
            steps += 1
            if len(buffer) > 500 and len(buffer) >= BATCH:
                batch = buffer.sample(BATCH)
                loss = agent.update(batch)
                losses.append(loss)
            if done or steps > MAX_STEPS_PER_EP:
                break
        total_reward += ep_reward

    # epsilon decay
    agent.eps = max(0.05, agent.eps * 0.95)
    avg_loss = np.mean(losses) if losses else 0
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | TotalReward: {total_reward:.2f} | AvgLoss: {avg_loss:.4f} | Eps: {agent.eps:.3f}")

# Save agent
torch.save(agent.q.state_dict(), MODEL_OUT)
print("Saved DQN agent to", MODEL_OUT)


Epoch 1/30 | TotalReward: 152.01 | AvgLoss: 0.0896 | Eps: 0.950
Epoch 2/30 | TotalReward: 149.46 | AvgLoss: 0.0935 | Eps: 0.902
Epoch 3/30 | TotalReward: 145.81 | AvgLoss: 0.1070 | Eps: 0.857
Epoch 4/30 | TotalReward: 157.11 | AvgLoss: 0.1190 | Eps: 0.815
Epoch 5/30 | TotalReward: 157.06 | AvgLoss: 0.1332 | Eps: 0.774
Epoch 6/30 | TotalReward: 169.71 | AvgLoss: 0.1453 | Eps: 0.735
Epoch 7/30 | TotalReward: 170.11 | AvgLoss: 0.1558 | Eps: 0.698
Epoch 8/30 | TotalReward: 159.06 | AvgLoss: 0.1657 | Eps: 0.663
Epoch 9/30 | TotalReward: 169.41 | AvgLoss: 0.1762 | Eps: 0.630
Epoch 10/30 | TotalReward: 173.76 | AvgLoss: 0.1803 | Eps: 0.599
Epoch 11/30 | TotalReward: 168.21 | AvgLoss: 0.1908 | Eps: 0.569
Epoch 12/30 | TotalReward: 179.76 | AvgLoss: 0.1932 | Eps: 0.540
Epoch 13/30 | TotalReward: 168.96 | AvgLoss: 0.1955 | Eps: 0.513
Epoch 14/30 | TotalReward: 184.66 | AvgLoss: 0.1989 | Eps: 0.488
Epoch 15/30 | TotalReward: 183.31 | AvgLoss: 0.2138 | Eps: 0.463
Epoch 16/30 | TotalReward: 177.06 

In [13]:
# Evaluate deterministic policy (eps=0)
agent.eps = 0.0
def evaluate(episodes, agent, max_steps=200):
    rewards = []
    for emb_seq, lab_seq in episodes:
        env = SimEnv(emb_seq, lab_seq, window_bad=10, goal_hold=3)
        s = env.reset()
        total_r = 0
        steps = 0
        while True:
            a = agent.select_action(s)
            s, r, done, info = env.step(a)
            total_r += r
            steps += 1
            if done or steps > max_steps:
                break
        rewards.append(total_r)
    return np.mean(rewards), np.std(rewards)

mean_r, std_r = evaluate(val_episodes, agent)
print("Validation reward mean:", mean_r, "std:", std_r)


Validation reward mean: 11.839999999999955 std: 5.372671588697727
